

<table width="100%" style="padding:0; margin-left:-6px;">
    <tr style="padding=0;" width="100%">
        <td width="auto">
           <hr style="border:2px solid darkblue">
            <h1> nb1. Query one source</h1>
           <hr style="border:2px solid darkblue">
        </td>
        <td width="60px">
           <img src="ALMAsmall.png" align="right"/> 
        </td>
    </tr>
</table>    
        

In this notebook we query one source in three modes: 
    
* By coordinate:
    * within a certain radius around a given coordinate (for point sources) 
    * within a certain radius around a given coordinate, estimating the distance to the archival sources
<br/><br/>
* By the resolved source name using [Sesame](http://cds.u-strasbg.fr/cgi-bin/Sesame)
<br/><br/>
    
* By the ALMA source name
    * full name   
    * part of the name
    <br/><br/>
    
-------------

The relevant columns in the [ALMA Science Archive TAP service](https://almascience.org/tap) are 
* *s_ra*, *s_dec* (RA and Dec in degrees)
* *s_region* (STC-S footprint of the observation)
* *target_name* (source name)
 
--------- ----- -----


First we need to set up the python modules

In [1]:
import numpy as np
import astropy
import astropy.coordinates
import pyvo
import pandas as pd

service = pyvo.dal.TAPService("https://almascience.eso.org/tap")      # for the EU ALMA TAP service

# service = pyvo.dal.TAPService("https://almascience.nao.ac.jp/tap")  # for the EA ALMA TAP service
# service = pyvo.dal.TAPService("https://almascience.nrao.edu/tap")   # for the NA ALMA TAP service

<hr style="border:2px solid darkblue"> </hr>

## By coordinate

<hr style="border:2px solid darkblue"> </hr>



The most basic search of a source is based on its coordinates. To do this we use the INTERSECT function to find all observations where the footprint (s_region) overlaps with the search circle. 

In [2]:
def query_coordinate(service, ra, dec, radius):
    """Queries ALMA footprints that intersect the search circle
    
       service        pyvo TAPService instance           
       ra             Right Ascension in decimal degrees
       dec            Declination in decimal degrees
       radius         Search radius in decimal degrees
           
       returns        pandas table
       """    
    
    query = f"""
            SELECT * 
            FROM ivoa.obscore   
            WHERE INTERSECTS(CIRCLE('ICRS',{ra},{dec},{radius}),s_region)=1  """
    
    return service.search(query).to_table().to_pandas()

<hr style="border:2px solid darkblue"> </hr>

## By coordinate: Display and sort by distance

<hr style="border:2px solid darkblue"> </hr>




It is also possible to add a column to the ADQL output. For example we can now display the distance of the positions in the archive catalogue to the searched coordinate and sort by it.

In [3]:
def query_coordinate_display_distance(service, ra, dec, radius):
    """Queries ALMA footprints that intersect the search circle
    
       service        pyvo TAPService instance           
       ra             Right Ascension in decimal degrees
       dec            Declination in decimal degrees
       radius         Search radius in decimal degrees
           
       returns        pandas table including the distance column
       """    

    query = f""" 
            SELECT DISTANCE(POINT('ICRS', s_ra, s_dec), POINT('ICRS',{ra},{dec})) AS dist, * 
            FROM ivoa.obscore 
            WHERE INTERSECTS(CIRCLE('ICRS',{ra},{dec},{radius}),s_region)=1 
            ORDER BY dist"""

    return service.search(query).to_table().to_pandas()

<hr style="border:2px solid darkblue"> </hr>

## By the resolved source name

<hr style="border:2px solid darkblue"> </hr>

Astropy comes with functionality to resolve astronomical source names into coordinates using the [Sesame](http://cds.u-strasbg.fr/cgi-bin/Sesame) service of [CDS](http://cds.u-strasbg.fr). This can be combined with the coordinate query above.

In [4]:
coordinates = astropy.coordinates.SkyCoord.from_name("Cen A")
print(coordinates.ra.degree, coordinates.dec.degree)

201.36506288 -43.01911267


<hr style="border:2px solid darkblue"> </hr>

## By the ALMA source name
<hr style="border:2px solid darkblue"> </hr>

These two functions encapsulate the search for a source name **as given by the PI in the proposal to ALMA** (for a search using a name-resolver, see above):

In [5]:
def query_alma_source_name_exact(service, source_name):
    """ queries the ALMA source name as given by the PI
    
       service        pyvo TAPService instance           
       source_name    complete source name 
           
       returns        pandas table
    """
    
    query = f"""  
            SELECT *  
            FROM ivoa.obscore  
            WHERE target_name = '{source_name}' """
    
    return service.search(query).to_table().to_pandas()


def query_alma_source_name_included(service, source_name):
    """ queries a portion of the ALMA source name as given by the PI
    
       service        pyvo TAPService instance           
       source_name    portion of the source name 
           
       returns        pandas table
    """   
    
    query = f""" 
            SELECT *  
            FROM ivoa.obscore  
            WHERE target_name LIKE '%{source_name}%'  """
    
    return service.search(query).to_table().to_pandas()

-----------------

## Example 1a: Find sources at coordinate RA: 201.365, DEC: -43.019

In [6]:
results = query_coordinate(service, 201.365, -43.019, 0.006)
results.head(5)

,access_url,access_format,proposal_id,data_rights,gal_longitude,gal_latitude,obs_publisher_did,obs_collection,facility_name,instrument_name,...,frequency,velocity_resolution,obs_creator_name,pub_title,first_author,qa2_passed,bib_reference,science_keyword,scientific_category,lastModified
0,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,...,238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
1,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,...,238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
2,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,...,238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
3,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,...,238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
4,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00010.S,Public,309.515914,19.417224,ADS/JAO.ALMA#2011.0.00010.S,ALMA,JAO,ALMA,...,219.025328,663.940713,"Ott, Juergen",ALMA Observations of the Physical and Chemical...,"McCoy, Mark",T,2017ApJ...851...76M,"Active Galactic Nuclei (AGN)/Quasars (QSO), Me...",Active galaxies,2021-09-30T16:34:41.133


The same query but now adding the distance column:

In [7]:
pd.set_option('display.max_columns', None)

results = query_coordinate_display_distance(service, 201.365, -43.019, 0.006)
results.head(5)

,dist,access_url,access_format,proposal_id,data_rights,gal_longitude,gal_latitude,obs_publisher_did,obs_collection,facility_name,instrument_name,obs_id,dataproduct_type,calib_level,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,pol_states,o_ucd,band_list,em_resolution,authors,pub_abstract,publication_year,proposal_abstract,schedblock_name,proposal_authors,sensitivity_10kms,cont_sensitivity_bandwidth,pwv,group_ous_uid,member_ous_uid,asdm_uid,obs_title,type,scan_intent,science_observation,spatial_scale_max,bandwidth,antenna_arrays,is_mosaic,obs_release_date,spatial_resolution,frequency_support,frequency,velocity_resolution,obs_creator_name,pub_title,first_author,qa2_passed,bib_reference,science_keyword,scientific_category,lastModified
0,1.484554,http://almascience.org/aq?member_ous_id=uid://...,text/html,2012.1.00225.S,Public,309.515896,19.417338,ADS/JAO.ALMA#2012.1.00225.S,ALMA,JAO,ALMA,uid://A002/X7d1738/X58,cube,2,Centaurus_a,201.365018,-43.019002,0.018076,Circle ICRS 201.365018 -43.019002 0.009038,1.170768,56815.888687,56815.901698,211.680,211.680,0.003338,0.003374,183907.921659,/XX/YY/,phot.flux.density;phys.polarization,3,613.938003,"Espada, D.; Matsushita, S.; Miura, R. E.; Isra...",Previous investigations have employed more tha...,2016,In powerful radio-galaxies the properties of t...,Cen_a_b3,"Wiklind, Tommy; Matsushita, Satoki; Impellizze...",1.940705,0.063029,1.475880,uid://A002/X7d1738/X57,uid://A002/X7d1738/X58,uid://A002/X839000/Xb6f,The Warm/Dense Circumnuclear Disk and Black Ho...,S,TARGET WVR,T,10.290043,9.375000e+08,A007:DV23 A010:DV12 A011:DV22 A016:DA62 A019:D...,F,2015-08-19T19:15:57.000,1.170768,"[88.00..88.94GHz,488.28kHz,1.9mJy/beam@10km/s,...",89.484582,1609.278532,"Espada, Daniel",ALMACAL I: First Dual-band Number Counts from ...,"Espada, D. Oteo, I. Thelen, Alexander E.",T,2016ApJ...822...36O 2017ApJ...843..136E 2019Ic...,"Active Galactic Nuclei (AGN)/Quasars (QSO), Ou...",Active galaxies,2021-09-30T16:34:41.133
1,1.484554,http://almascience.org/aq?member_ous_id=uid://...,text/html,2012.1.00225.S,Public,309.515896,19.417338,ADS/JAO.ALMA#2012.1.00225.S,ALMA,JAO,ALMA,uid://A002/X7d1738/X58,cube,2,Centaurus_a,201.365018,-43.019002,0.018076,Circle ICRS 201.365018 -43.019002 0.009038,1.170768,56815.888687,56815.901698,211.680,211.680,0.003371,0.003407,182136.313740,/XX/YY/,phot.flux.density;phys.polarization,3,613.938003,"Espada, D.; Matsushita, S.; Miura, R. E.; Isra...",Previous investigations have employed more tha...,2016,In powerful radio-galaxies the properties of t...,Cen_a_b3,"Wiklind, Tommy; Matsushita, Satoki; Impellizze...",1.936746,0.063029,1.475880,uid://A002/X7d1738/X57,uid://A002/X7d1738/X58,uid://A002/X839000/Xb6f,The Warm/Dense Circumnuclear Disk and Black Ho...,S,TARGET WVR,T,10.290043,9.375000e+08,A007:DV23 A010:DV12 A011:DV22 A016:DA62 A019:D...,F,2015-08-19T19:15:57.000,1.170768,"[88.00..88.94GHz,488.28kHz,1.9mJy/beam@10km/s,...",89.484582,1609.278532,"Espada, Daniel",ALMACAL I: First Dual-band Number Counts from ...,"Espada, D. Oteo, I. Thelen, Alexander E.",T,2016ApJ...822...36O 2017ApJ...843..136E 2019Ic...,"Active Galactic Nuclei (AGN)/Quasars (QSO), Ou...",Active galaxies,2021-09-30T16:34:41.133
2,1.484554,http://almascience.org/aq?member_ous_id=uid://...,text/html,2012.1.00225.S,Public,309.515896,19.417338,ADS/JAO.ALMA#2012.1.00225.S,ALMA,JAO,ALMA,uid://A002/X7d1738/X58,cube,2,Centaurus_a,201.365018,-43.019002,0.018076,Circle ICRS 201.365018 -43.019002 0.009038,1.170768,56815.888687,56815.901698,211.680,211.680,0.003296,0.003330,186289.975282,/XX/YY/,phot.flux.density;phys.polarization,3,613.938003,"Espada, D.; Matsushita, S.; Miura, R. E.; Isra...",Previous investigations have employed more tha...,2016,In powerful radio-galaxies the properties of t...,Cen_a_b3,"Wiklind, Tommy; Matsushita, Satoki; Impellizze...",1.928636,0.063029,1.475880,uid://A002/X7d1738/X57,uid://A002/X7d1738/X58,uid://A002/X839000/Xb6f,The Warm

-----------------

## Example 1b: Query the source Centaurus A using Sesame

Many sources have a number of names and you can not be sure that the source name the PI placed into the ALMA proposal is the source name you have in mind. We use the name-resolver [Sesame](http://cds.u-strasbg.fr/cgi-bin/Sesame) provided by [CDS](http://cds.u-strasbg.fr) to first resolve the source name into coordinates and then search around those coordinates.

In [8]:
coordinates = astropy.coordinates.SkyCoord.from_name("Cen A")

In [9]:
query_coordinate(service, coordinates.ra.degree, coordinates.dec.degree, 0.006).head(5)

,access_url,access_format,proposal_id,data_rights,gal_longitude,gal_latitude,obs_publisher_did,obs_collection,facility_name,instrument_name,obs_id,dataproduct_type,calib_level,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,pol_states,o_ucd,band_list,em_resolution,authors,pub_abstract,publication_year,proposal_abstract,schedblock_name,proposal_authors,sensitivity_10kms,cont_sensitivity_bandwidth,pwv,group_ous_uid,member_ous_uid,asdm_uid,obs_title,type,scan_intent,science_observation,spatial_scale_max,bandwidth,antenna_arrays,is_mosaic,obs_release_date,spatial_resolution,frequency_support,frequency,velocity_resolution,obs_creator_name,pub_title,first_author,qa2_passed,bib_reference,science_keyword,scientific_category,lastModified
0,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001221,0.001231,251315.70776,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.319542,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
1,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001212,0.001221,253235.70776,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.283133,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
2,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001297,0.001308,236599.53752,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.238502,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13

-----------------


## Example 1c: Query the ALMA source name "Centaurus"

Although in most cases, using the name resolver as above is the best way to search for source names, there can be occasions where searching for the name as given by the PI is useful. (Note that SQL queries are case-sensitive).

In [10]:
query_alma_source_name_exact(service, "Centaurus A")

,access_url,access_format,proposal_id,data_rights,gal_longitude,gal_latitude,obs_publisher_did,obs_collection,facility_name,instrument_name,obs_id,dataproduct_type,calib_level,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,pol_states,o_ucd,band_list,em_resolution,authors,pub_abstract,publication_year,proposal_abstract,schedblock_name,proposal_authors,sensitivity_10kms,cont_sensitivity_bandwidth,pwv,group_ous_uid,member_ous_uid,asdm_uid,obs_title,type,scan_intent,science_observation,spatial_scale_max,bandwidth,antenna_arrays,is_mosaic,obs_release_date,spatial_resolution,frequency_support,frequency,velocity_resolution,obs_creator_name,pub_title,first_author,qa2_passed,bib_reference,science_keyword,scientific_category,lastModified
0,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001221,0.001231,251315.70776,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.319542,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
1,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001212,0.001221,253235.70776,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.283133,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
2,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001297,0.001308,236599.53752,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.238502,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13

-----------------

## Example 1d: Query the source name "Centaurus"
Since source names in the ALMA proposals might be spelled differently it can be useful to search for parts of the name.

In [11]:
table = query_alma_source_name_included(service, "ntaur")
table.head(5) ### showing only first 5 rows

,access_url,access_format,proposal_id,data_rights,gal_longitude,gal_latitude,obs_publisher_did,obs_collection,facility_name,instrument_name,obs_id,dataproduct_type,calib_level,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,pol_states,o_ucd,band_list,em_resolution,authors,pub_abstract,publication_year,proposal_abstract,schedblock_name,proposal_authors,sensitivity_10kms,cont_sensitivity_bandwidth,pwv,group_ous_uid,member_ous_uid,asdm_uid,obs_title,type,scan_intent,science_observation,spatial_scale_max,bandwidth,antenna_arrays,is_mosaic,obs_release_date,spatial_resolution,frequency_support,frequency,velocity_resolution,obs_creator_name,pub_title,first_author,qa2_passed,bib_reference,science_keyword,scientific_category,lastModified
0,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001221,0.001231,251315.707760,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.319542,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
1,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001212,0.001221,253235.707760,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.283133,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz,13.2mJy/beam@10km...",238.261613,1183.847494,"observatory, ALMA",ALMA Observations of the Physical and Chemical...,"Azeez, Jazeel H. Israel, F. P. McCoy, Mark",T,2014A&A...562A..96I 2017AdAst2017E...6A 2017Ap...,"Galactic centres/nuclei, Active Galactic Nucle...",Active galaxies,2021-09-30T16:34:41.133
2,http://almascience.org/aq?member_ous_id=uid://...,text/html,2011.0.00008.SV,Public,309.512363,19.416239,ADS/JAO.ALMA#2011.0.00008.SV,ALMA,JAO,ALMA,uid://A002/X259150/X157,cube,2,Centaurus A,201.360719,-43.020553,0.058089,Polygon ICRS 201.401254 -43.032096 201.401354 ...,1.023315,55784.795689,55785.043473,344.261,344.261,0.001297,0.001308,236599.537520,/XX/YY/,phot.flux.density;phys.polarization,6,306.959997,"Azeez, Jazeel H.; Abidin, Zamri Z.; Hwang, C. ...","Centaurus A, with its gas-rich elliptical host...",2014,Science Verification (SV) is the process by wh...,CenA mosaic,,13.238502,0.432702,1.663192,,uid://A002/X259150/X157,uid://A002/X2762c7/X132,Science verification observation of Centaurus A,SV,TARGET WVR,T,9.714115,1.875000e+09,A004:DV07 A008:DV09 A009:DA41 A011:DV12 A015:D...,T,2016-06-24T14:02:07.000,1.023315,"[229.20..231.08GHz,976.56kHz